# 데이터 사이언스 특론 5주차 코드 및 내용 복습


## 1. Tidy Data (깔끔한 데이터)


*   **Tidy Data**는 분석에 적합한 구조로 정리된 데이터를 의미한다.
    * 각 **변수(variable)** 는 하나의 **열(column)** 을 구성
    * 각 **관측치(observation)** 는 하나의 **행(row)** 을 구성
    * 각 **관측 단위(observational unit)** 는 하나의 **테이블**을 구성
*   Tidy Data가 아닌 데이터를 **Messy Data** 라고 하며, 실제 수집된 데이터는 대부분 messy하다.



**<Messy Data의 3가지 대표 유형>**
1.   열 이름이 변수가 아니라 **값**인 경우
2.   **여러 변수**가 하나의 열에 저장된 경우
3.   변수가 **행과 열 모두**에 저장된 경우


## 2. 핵심 함수



*   **melt()** : Wide → Long 변환. 열에 흩어진 값들을 하나의 열로 모아준다.
    * `id_vars` : 그대로 유지할 식별 열 (녹여지지 않는 열)
    * `var_name` : 녹여진 열 이름들이 들어갈 새 열의 이름
    * `value_name` : 녹여진 값들이 들어갈 새 열의 이름



*   **pivot_table()** : Long → Wide 변환. melt()의 반대 방향.
    * `values` : 셀에 들어갈 값
    * `index` : 행 인덱스가 될 열
    * `columns` : 열 헤더가 될 열



*   **apply()** : 각 원소에 함수를 적용하여 새로운 열을 만들 때 사용
    * lambda와 함께 간단한 변환, 사용자 정의 함수와 함께 복잡한 변환을 수행


In [ ]:
import numpy as np
import pandas as pd


## 3. Data Structure -> 어떤 구조가 Tidy한가?


같은 데이터도 구조에 따라 분석 용이성이 달라진다.  
아래 두 표현 중 어떤 것이 tidy data인지 비교해보자.


In [ ]:
df = (pd.DataFrame([['John Smith', np.nan, 2],
                    ['Jane Doe', 16, 11],
                    ['Mary Johnson', 3, 1]],
                   columns=['Name', 'Treatment A', 'Treatment B']))



*   **표현 1** : 사람별로 Treatment A, B가 열에 있는 Wide format


In [ ]:
df



*   **표현 2** : 전치(Transpose)한 형태


In [ ]:
df.set_index('Name').T


위 두 표현 모두 **messy**하다.  
Treatment A/B가 열 이름에 들어가 있기 때문이다.  
→ `melt()`를 사용해서 Tidy하게 변환해야 한다.



*   **melt()로 Tidy하게 변환**


In [ ]:
df_tidy = df.melt(id_vars=['Name'], value_vars=['Treatment A', 'Treatment B'])
df_tidy


`Name`은 그대로 유지(id_vars), Treatment A/B 열이 녹아서 `variable`과 `value` 두 열로 변환됨.  
이제 각 행이 "한 사람의 한 치료법에 대한 한 관측치"가 되어 **tidy** 해진다.


 
## 4. Case 1: 열 이름이 변수가 아니라 값인 경우


*   Pew Research Center의 종교별 소득 분포 데이터
*   소득 구간(`<$10k`, `$10-20k`, ...)이 **열 이름**으로 되어 있어 messy한 상태
*   이 소득 구간들은 변수의 **값**이지, 변수 **이름**이 아니다.


In [ ]:
pew = pd.read_csv('pew.txt', sep='\t')  # tab separated values
pew.head(10)



*   **melt()로 소득 구간을 하나의 열로 모으기**


In [ ]:
pew_melt = pew.melt(id_vars=['religion'], var_name='income', value_name='freq')
pew_melt[pew_melt['religion'] == 'Agnostic']


`religion`은 식별 열로 유지하고, 나머지 열들이 `income` 열의 값으로 녹아들어간다.  
이제 각 행이 "하나의 종교 × 하나의 소득 구간"에 대한 관측치가 된다.


 
## 5. Case 2: 여러 변수가 하나의 열에 저장된 경우


*   결핵(TB) 데이터: 열 이름에 **성별과 나이**가 함께 인코딩되어 있음
*   예: `new_sp_m04` → 남성(m) + 0~4세(04)


In [ ]:
tb = pd.read_csv('tb.csv')
tb[['iso2', 'year', 'new_sp_m04', 'new_sp_m514', 'new_sp_m1524', 'new_sp_m2534', 'new_sp_m3544', 'new_sp_m4554',
    'new_sp_m5564', 'new_sp_m65', 'new_sp_mu', 'new_sp_f04']].head(10)



*   **Step 1: 열 이름 정리** — `new_sp_` 접두사 제거


In [ ]:
# 'new_sp_' 접두사 제거
tb.columns = [column.replace('new_sp_', '') for column in tb.columns.tolist()]
tb.drop(['new_sp'], axis=1, inplace=True)
tb.head(10)



*   **Step 2: melt()로 Long format 변환**


In [ ]:
tb_molten = tb.melt(id_vars=['iso2', 'year'], var_name='sex_and_age', value_name='cases')
tb_molten.head(10)



*   **Step 3: 필터링 + 정렬 확인**


In [ ]:
(tb_molten.where((tb_molten['sex_and_age'] != 'm04') &
                (tb_molten['sex_and_age'] != 'm514') &
                (tb_molten['sex_and_age'] != 'u') &
                (tb_molten['year'] == 2000))
          .dropna()
          .sort_values(['iso2', 'sex_and_age'])
          .head(10)
)



*   **Step 4: 성별(sex) 열 분리** — 첫 글자만 추출 (lambda 사용)


In [ ]:
# 성별: 첫 글자만 추출
tb_molten['sex'] = tb_molten['sex_and_age'].apply(lambda v: v[0])
tb_molten



*   **Step 5: 나이(age) 열 분리** — 인코딩 규칙이 불규칙하므로 사용자 정의 함수 필요


In [ ]:
def get_age(v):
    # handle exceptions first
    if v[1:] == 'u':
        return 'unknown'
    elif v[1:] == '04':
        return '0-4'
    elif v[1:] == '014':
        return '0-14'
    elif v[1:] == '514': 
        return '5-14'
    elif v[1:] == '65':
        return '65+'
    else:
        # when both staring and ending ages are two-digit numbers
        return v[1:3] + '-' + v[3:]
    
tb_molten['sex'] = tb_molten['sex_and_age'].apply(lambda v: v[0])
tb_molten['age'] = tb_molten['sex_and_age'].apply(get_age)
tb_molten['age'].unique()


`apply()` + `lambda` : 단순한 변환 (성별 추출)에 사용  
`apply()` + 사용자 정의 함수 : 복잡한 조건 분기가 필요한 변환 (나이 추출)에 사용



*   **Step 6: 불필요한 열 제거 + 열 순서 정리 → Tidy Data 완성**


In [ ]:
tb_molten.drop('sex_and_age', axis=1, inplace=True)
tb_molten = tb_molten[['iso2', 'year', 'sex', 'age', 'cases']]
tb_molten[tb_molten['year'] == 2000].head(10)


이제 각 행이 "국가 × 연도 × 성별 × 나이대"에 대한 한 관측치가 되어 tidy해졌다.


In [ ]:
# 필터링 + 정렬 예시
(tb_molten[(tb_molten['year'] == 2000) & (tb_molten['age'] != 'unknown')].sort_values(['iso2'])
          .head(10)
)


 
## 6. Case 3: 변수가 행과 열 모두에 저장된 경우


*   날씨 데이터: 날짜(`d1`, `d2`, ...)가 열에, 변수(`tmax`, `tmin`)가 행에 저장되어 있음
*   `melt()` + `pivot_table()` 조합으로 정리해야 한다.


In [ ]:
weather = pd.read_csv('weather.txt', sep='\t', na_values=['---'])
weather.head(10)


`d1`, `d2`, ... `d31` 열은 날짜를 의미 → 열 이름이 값인 경우 (Case 1과 유사)  
`element` 열에 `tmax`, `tmin`이 있음 → 이 값들은 별도의 **열**이어야 함



*   **Step 1: melt()로 날짜 열을 녹이기**


In [ ]:
weather_molten = weather.melt(['id', 'year', 'month', 'element'])
weather_molten.head()



*   **Step 2: day 열 추출** — `'d1'` → `'1'` 로 변환 후 결측치 제거


In [ ]:
# 'd1' → '1', 'd2' → '2' 등으로 변환
weather_molten['day'] = weather_molten['variable'].apply(lambda v: v[1:])
weather_molten.dropna(inplace=True)
weather_molten.head()



*   **Step 3: 날짜(date) 열 생성** — `pd.to_datetime()`에 딕셔너리로 year/month/day 전달


In [ ]:
# year, month, day를 합쳐서 datetime 열 생성
weather_molten['date'] = pd.to_datetime(dict(year=weather_molten['year'],
                                             month=weather_molten['month'],
                                             day=weather_molten['day']))
weather_molten.head()



*   **Step 4: 불필요한 열 제거 + 열 순서 정리**


In [ ]:
weather_molten.drop(['year', 'month', 'day', 'variable'], axis=1, inplace=True)
# change the order of columns
weather_molten = weather_molten[['id', 'date', 'element', 'value']]
weather_molten.head()



*   **Step 5: pivot_table()로 element를 열로 펼치기 → Tidy Data 완성**


`element` 열에 있는 `tmax`, `tmin`을 **각각의 열**로 만들어야 tidy해진다.  
이때 `melt()`의 반대인 `pivot_table()`을 사용한다.


In [ ]:
weather_tidy = weather_molten.pivot_table(values='value', index=['id', 'date'], columns=['element']).reset_index()
weather_tidy.head()


`pivot_table()`은 `melt()`의 **역연산** — Long → Wide 변환  
`reset_index()` : multi-index를 다시 일반 열로 되돌린다.


 
## 7. 종합 정리



**<Messy 유형별 변환 흐름>**

| Messy 유형 | 핵심 함수 | 핵심 작업 |
|---|---|---|
| 열 이름이 값인 경우 | `melt()` | 열을 값으로 녹이기 (Wide → Long) |
| 한 열에 여러 변수 | `melt()` + `apply()` | 녹인 후 문자열 파싱으로 변수 분리 |
| 변수가 행+열 모두에 | `melt()` + `pivot_table()` | 녹인 후 다시 펼치기 |



**<melt() vs pivot_table()>**

| | `melt()` | `pivot_table()` |
|---|---|---|
| 방향 | Wide → Long | Long → Wide |
| 역할 | 열을 행으로 녹이기 | 행을 열로 펼치기 |
| 사용 시점 | 열 이름에 값이 들어가 있을 때 | 하나의 열에 있는 값이 별도 열이어야 할 때 |



**<주요 함수 요약>**
*   `pd.melt(id_vars, var_name, value_name)` : 열을 행으로 변환
*   `df.pivot_table(values, index, columns)` : 행을 열로 변환
*   `df.apply(func)` : 각 원소에 함수 적용 (lambda 또는 사용자 정의 함수)
*   `pd.to_datetime(dict(...))` : year, month, day 딕셔너리로 날짜 생성
*   `df.drop(columns, axis=1, inplace=True)` : 불필요한 열 제거
*   `df.dropna(inplace=True)` : 결측치 행 제거
*   list comprehension으로 열 이름 일괄 정리 : `[col.replace('old', '') for col in df.columns]`


 
## 8. 통계에서의 변수 유형 (Types of Variables)



*   **연속형 변수 (Continuous variables)**
    * 숫자로 측정 가능하며, 무한히 많은 값을 가질 수 있는 변수
    * e.g., 키, 몸무게, 온도, 시간



*   **범주형 변수 (Categorical variables)**
    * **Nominal (명목형)** : 본질적인 순서가 없는 두 개 이상의 범주
        * e.g., 브랜드 명, 혈액형
    * **Dichotomous (이분형)** : 두 가지 범주만 있는 명목형
        * e.g., 성별, yes/no
    * **Ordinal (순서형)** : 순서를 정하거나 순위를 매길 수 있는 범주
        * e.g., 학점 성적 (A > B > C)



## 9. 통계의 분류



*   **기술 통계 (Descriptive Statistics)**
    * 현재 가지고 있는 데이터의 특성을 **요약 및 설명**
    * 이 결과를 데이터 범위를 넘어서까지 일반화하는 것에는 어려움이 있음
    * → **탐색적 데이터 분석 (EDA)**



*   **추론 통계 (Inferential Statistics)**
    * 우리가 가진 데이터를 바탕으로, 데이터 너머에 있는 **모집단에 대한 추론**을 가능하게 함
    * e.g., t-test
    * 예측(prediction) 모델, 기계 학습(ML)을 구축하는 기반이 됨



## 10. EDA (Exploratory Data Analysis) — 탐색적 데이터 분석



*   John W. Tukey (1977)가 제안
*   데이터를 **시각화하고 요약**하여 데이터로부터 **통찰(insights)** 을 발견하는 기술
    * 보이지 않는 가설을 확인하는 것보다, **예상치 못한 것을 찾아내는 것**이 분석의 즐거움
    * 단순한 계산이 아닌 **"데이터와의 대화"**
*   주요 예시
    * 5-number summary: 최댓값, 최솟값, 중앙값 및 사분위수
    * Box plots
    * Stem and leaf diagrams



*   **기술 통계 또는 요약 통계**
    * 소수의 통계량이나 도표를 사용하여 데이터의 전반적인 특징을 설명
*   수치적 지표
    * Number of samples, average, mode(최빈값), variance, standard deviation, 5-number summary, ...
*   차트 종류 (one variable)
    * Dot plot, jitter plot, error plot, box-and-whisker plot, histogram, cumulative distribution function, ...
*   차트 종류 (two or more variables)
    * Stacked plots, parallel coordinate plot, ...



*   **EDA sample case — 이 회사에 투자할 것인가?**


고객 당 매출(Rev/Customer) 데이터만 보면 $5.00 → $4.50 → $4.33 → $4.25 → $4.50으로 안정적으로 보인다.  
그러나 **코호트(cohort) 분석**을 하면, 실제로는 각 코호트의 매출이 시간이 지남에 따라 급격히 감소하고 있으며,  
신규 고객의 첫 달 매출이 점점 높아지면서 평균이 유지되는 것처럼 **착시**를 만들고 있었다.  
→ 같은 데이터도 **어떤 관점(축)으로 분석하느냐**에 따라 완전히 다른 결론이 나올 수 있다.


 
## 11. 분산과 표준편차


데이터 값들이 **평균으로부터 얼마나 떨어져 있는지**, 즉 데이터의 **퍼짐 정도(산포도)** 를 수치화한 것.

*   **평균 (Mean)**
    * x̄ = (1/n) × Σxᵢ
*   **분산 (Variance)**
    * Var(x) = σ² = (1/n) × Σ(xᵢ - x̄)²
    * 편차의 제곱 평균
*   **표준편차 (Standard Deviation)**
    * σ = √Var(x)
    * 분산의 제곱근. 원래 데이터와 같은 단위를 가짐.
    * 표준편차가 작으면 → 데이터가 평균 주위에 밀집
    * 표준편차가 크면 → 데이터가 넓게 퍼져 있음


 
## 12. 공분산과 상관관계 (Covariance and Correlation)


두 변수(xᵢ, yᵢ)가 **함께 변하는 정도**를 나타내는 지표.

*   **공분산 (Covariance)**
    * 두 변수가 같은 방향으로 변하는지, 반대 방향으로 변하는지
    * cov(X,Y) = (1/n) × Σ(xᵢ - x̄)(yᵢ - ȳ)
    * 양수: 같은 방향, 음수: 반대 방향
    * 단위에 따라 값의 크기가 달라져서 비교가 어렵다.



*   **상관계수 (Correlation)**
    * 공분산을 각 변수의 표준편차로 나누어 **(-1, 1)로 정규화**한 값
    * corr(X,Y) = cov(X,Y) / (σₓ × σᵧ)
    * 데이터의 단위를 제거하여 동일한 기준으로 비교 가능
    * 절대값이 1에 가까우면 "관계가 있다"
    * **선형의 상관관계만** 비교 가능 (비선형 관계는 상관계수가 0으로 나올 수 있음)


 
## 13. 상관관계와 인과관계 (Correlation vs. Causation)



*   **상관관계 (Correlated)**
    * 두 변수가 관련되어 있는 것 (다른 무언가에 의존할 수도 있음)
*   **인과관계 (Causal)**
    * 하나의 독립변수가 다른 종속변수에 **직접적으로** 영향을 줄 때
*   인과관계가 있다면, 분명히 상관관계는 있다.
*   그러나 **상관관계가 있다고 반드시 인과관계가 있는 것은 아니다!**



*   **허위 상관관계 (Spurious Correlations)**
    * Apple 특허 수 ↔ LA 다저스 티켓 판매 (r=0.968)
    * "never gonna give you up" 검색량 ↔ Tesla 주가 (r=0.945)
    * 높은 상관계수(r)와 유의한 p-value가 나와도 **인과관계를 의미하지 않는다.**
    * 참고: https://www.tylervigen.com/


 
## 14. Reading list

*   Tidy data
    * Paper: https://vita.had.co.nz/papers/tidy-data.pdf
    * Tidy data for R: http://r4ds.had.co.nz/tidy-data.html
*   Spurious correlations
    * https://www.tylervigen.com/
